Construimos la **Frontera Eficiente de Media-Varianza** de Markowitz (1952). Dado un conjunto de activos, la frontera describe el set de portafolios que minimizan la varianza para cada nivel de retorno esperado. Calculamos los parámetros A, B, C, D de la frontera, la cartera de mínima varianza, y las carteras generadoras **g** y **h**.

In [1]:
#| label: setup-fe
#| code-fold: true
#| code-summary: "Librerías y datos"

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import yfinance as yf

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

TICKERS = ['AAPL','MSFT','AMZN','GOOGL','META','JPM','BAC','GS','WFC','MS',
           'JNJ','PFE','UNH','MRK','ABBV','XOM','CVX','KO','PG','WMT']
RF_MONTHLY = 0.0448 / 12

raw = yf.download(TICKERS, start='2015-01-01', end='2024-12-31',
                  interval='1mo', auto_adjust=True, progress=False)['Close']
returns = raw.dropna(axis=1, thresh=int(0.9*len(raw))).pct_change().dropna()

mu = returns.mean().values          # vector de medias (N,)
V  = returns.cov().values           # matriz de covarianza (N×N)
N  = len(mu)
iota = np.ones(N)

print(f"Activos: {N}")
print(f"Observaciones: {len(returns)}")

Activos: 20
Observaciones: 119


### Parámetros de la frontera eficiente

A partir de la solución del problema de optimización de mínima varianza, los cuatro parámetros escalares de la frontera son:

$$A = \iota'V^{-1}\mu \qquad B = \mu'V^{-1}\mu \qquad C = \iota'V^{-1}\iota \qquad D = BC - A^2$$

La varianza de la cartera con retorno esperado $\mu_w$ en la frontera es:

$$\sigma^2_w = \frac{1}{C} + \frac{C}{D}\left(\mu_w - \frac{A}{C}\right)^2$$

In [2]:
#| label: parametros-frontera

V_inv = np.linalg.inv(V)

A = iota @ V_inv @ mu
B = mu   @ V_inv @ mu
C = iota @ V_inv @ iota
D = B * C - A**2

# Cartera de mínima varianza global (GMVP)
mu_gmvp    = A / C
var_gmvp   = 1 / C
sigma_gmvp = np.sqrt(var_gmvp)

print("Parámetros de la frontera eficiente")
print("-" * 38)
print(f"A = {A:.6f}")
print(f"B = {B:.6f}")
print(f"C = {C:.6f}")
print(f"D = {D:.6f}")
print()
print("Cartera de Mínima Varianza Global (GMVP)")
print("-" * 38)
print(f"Retorno esperado mensual: {mu_gmvp:.4%}")
print(f"Varianza:                 {var_gmvp:.6f}")
print(f"Desv. estándar mensual:   {sigma_gmvp:.4%}")

Parámetros de la frontera eficiente
--------------------------------------
A = 12.357688
B = 0.340227
C = 993.648909
D = 185.353321

Cartera de Mínima Varianza Global (GMVP)
--------------------------------------
Retorno esperado mensual: 1.2437%
Varianza:                 0.001006
Desv. estándar mensual:   3.1724%


### Carteras generadoras g y h

Cualquier cartera eficiente puede expresarse como combinación lineal de dos carteras base **g** y **h**:

$$w = g + h \cdot \mu_w$$

$$g = \frac{BV^{-1}\iota - AV^{-1}\mu}{D}, \qquad h = \frac{CV^{-1}\mu - AV^{-1}\iota}{D}$$

**Verificación:** $\iota'g = 1$ y $\iota'h = 0$ (g es una cartera válida; h es un portafolio de suma cero).

In [3]:
#| label: carteras-gh

g = (B * (V_inv @ iota) - A * (V_inv @ mu)) / D
h = (C * (V_inv @ mu)   - A * (V_inv @ iota)) / D

print("Verificación de carteras g y h:")
print(f"  ι'g = {iota @ g:.8f}  (debe ser 1.0)")
print(f"  ι'h = {iota @ h:.8f}  (debe ser 0.0)")
print()

# Verificar que g + h*mu_gmvp == GMVP weights
w_gmvp = g + h * mu_gmvp
w_gmvp_direct = (V_inv @ iota) / C
print(f"Consistencia GMVP (norma diferencia): {np.linalg.norm(w_gmvp - w_gmvp_direct):.2e}")

# Top asignaciones en GMVP
tickers_avail = returns.columns.tolist()
gmvp_df = pd.Series(w_gmvp_direct, index=tickers_avail).sort_values(ascending=False)
print("\nTop 5 pesos en la cartera de mínima varianza:")
print(gmvp_df.head(5).apply(lambda x: f'{x:.2%}').to_string())

Verificación de carteras g y h:
  ι'g = 1.00000000  (debe ser 1.0)
  ι'h = 0.00000000  (debe ser 0.0)

Consistencia GMVP (norma diferencia): 1.03e-16

Top 5 pesos en la cartera de mínima varianza:
PG       30.80%
JPM      23.83%
XOM      14.48%
GOOGL    13.01%
UNH      12.61%


### Frontera eficiente y cartera tangente

La **cartera tangente** (o cartera $q$) es el portafolio en la frontera eficiente que maximiza la razón de Sharpe. Es el punto donde la Capital Market Line (CML) es tangente a la frontera. Cualquier inversionista racional debería combinar esta cartera con el activo libre de riesgo.

$$w_q = \frac{V^{-1}(\mu - R_f \iota)}{\iota' V^{-1}(\mu - R_f \iota)}$$

In [4]:
#| label: frontera-grafico
#| code-fold: true
#| fig-cap: "Frontera eficiente, cartera de mínima varianza y cartera tangente"

# Frontera eficiente
mu_range = np.linspace(mu_gmvp * 0.5, mu.max() * 1.3, 300)
sigma_frontier = np.sqrt(1/C + (C/D) * (mu_range - A/C)**2)

# Cartera tangente
excess = mu - RF_MONTHLY
z_q = V_inv @ excess
w_q = z_q / (iota @ z_q)
mu_q    = mu @ w_q
sigma_q = np.sqrt(w_q @ V @ w_q)
sr_q    = (mu_q - RF_MONTHLY) / sigma_q

# CML
sigma_cml = np.linspace(0, sigma_q * 2, 200)
mu_cml    = RF_MONTHLY + sr_q * sigma_cml

# Activos individuales
sigma_assets = returns.std().values

fig, ax = plt.subplots(figsize=(11, 7))

# Parte ineficiente (debajo de GMVP)
mask_inef = mu_range < mu_gmvp
ax.plot(sigma_frontier[mask_inef]*100, mu_range[mask_inef]*100,
        color='#94a3b8', linewidth=1.5, linestyle='--', label='Frontera ineficiente')
ax.plot(sigma_frontier[~mask_inef]*100, mu_range[~mask_inef]*100,
        color='#2563eb', linewidth=2.5, label='Frontera eficiente')

ax.plot(sigma_cml*100, mu_cml*100, color='#16a34a', linewidth=1.5,
        linestyle='-.', label='Capital Market Line (CML)')

ax.scatter(sigma_gmvp*100, mu_gmvp*100, color='#dc2626', s=100, zorder=5,
           label=f'Mín. varianza: μ={mu_gmvp:.3%}, σ={sigma_gmvp:.3%}')
ax.scatter(sigma_q*100, mu_q*100, color='#f59e0b', s=120, marker='*', zorder=5,
           label=f'Cartera tangente: SR={sr_q:.3f}')

ax.scatter(sigma_assets*100, mu*100, color='#64748b', s=40, alpha=0.6,
           label='Activos individuales')
for i, t in enumerate(tickers_avail):
    ax.annotate(t, (sigma_assets[i]*100, mu[i]*100),
                fontsize=7, alpha=0.7, xytext=(3, 2), textcoords='offset points')

ax.axhline(RF_MONTHLY*100, color='gray', linestyle=':', alpha=0.5)
ax.text(0.02, RF_MONTHLY*100+0.01, f'Rf = {RF_MONTHLY:.4%}', fontsize=9, color='gray')

ax.set_xlabel('Desviación estándar mensual (%)', fontsize=12)
ax.set_ylabel('Retorno esperado mensual (%)', fontsize=12)
ax.set_title('Frontera Eficiente de Media-Varianza — S&P 500 (2015–2024)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

print(f"\nCartera tangente:")
print(f"  Retorno esperado: {mu_q:.4%}/mes")
print(f"  Desv. estándar:   {sigma_q:.4%}/mes")
print(f"  Razón de Sharpe:  {sr_q:.4f}")


Cartera tangente:
  Retorno esperado: 3.4007%/mes
  Desv. estándar:   5.9166%/mes
  Razón de Sharpe:  0.5117
